<a href="https://colab.research.google.com/github/2403A51L41/ReinforcementLearning/blob/main/RL_Assign11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "provenance": []
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [
    {
      "cell_type": "code",
      "source": [
        "gamma = 0.9\n",
        "theta = 1e-10\n",
        "\n",
        "states = [\"S0\", \"S1\", \"G\"]\n",
        "non_terminal_states = [\"S0\", \"S1\"]\n",
        "actions = [\"up\", \"down\", \"left\", \"right\"]\n",
        "\n",
        "# Grid layout:\n",
        "# S0   S1\n",
        "# WALL G\n",
        "\n",
        "grid = [\n",
        "    [\"S0\", \"S1\"],\n",
        "    [\"WALL\", \"G\"]\n",
        "]\n",
        "\n",
        "positions = {\n",
        "    \"S0\": (0, 0),\n",
        "    \"S1\": (0, 1),\n",
        "    \"G\": (1, 1)\n",
        "}\n",
        "\n",
        "wall = (1, 0)\n",
        "\n",
        "deltas = {\n",
        "    \"up\": (-1, 0),\n",
        "    \"down\": (1, 0),\n",
        "    \"left\": (0, -1),\n",
        "    \"right\": (0, 1)\n",
        "}\n",
        "\n",
        "print(\"STATES:\", states)\n",
        "print(\"ACTIONS:\", actions)\n",
        "print(\"Discount factor:\", gamma)\n",
        "print(\"Rewards: Normal = -1, Wall = -1, Goal = +10\")\n",
        "\n",
        "# Build transition and reward mappings\n",
        "transition = {}\n",
        "reward = {}\n",
        "\n",
        "for s in non_terminal_states:\n",
        "    r, c = positions[s]\n",
        "\n",
        "    for a in actions:\n",
        "        dr, dc = deltas[a]\n",
        "        nr, nc = r + dr, c + dc\n",
        "\n",
        "        if (not (0 <= nr < 2 and 0 <= nc < 2)\n",
        "                or (nr, nc) == wall):\n",
        "            next_state = s\n",
        "            immediate_reward = -1\n",
        "        else:\n",
        "            next_state = grid[nr][nc]\n",
        "            immediate_reward = 10 if next_state == \"G\" else -1\n",
        "\n",
        "        transition[(s, a)] = next_state\n",
        "        reward[(s, a)] = immediate_reward\n",
        "\n",
        "# Verify transitions\n",
        "print(\"\\nTRANSITION MAPPING\")\n",
        "for s in non_terminal_states:\n",
        "    for a in actions:\n",
        "        print(\n",
        "            f\"State: {s}, Action: {a}, \"\n",
        "            f\"Next State: {transition[(s, a)]}, \"\n",
        "            f\"Reward: {reward[(s, a)]}\"\n",
        "        )\n",
        "\n",
        "assert len(transition) == len(non_terminal_states) * len(actions)\n",
        "\n",
        "# Initialize state values\n",
        "V = {s: 0.0 for s in states}\n",
        "\n",
        "print(\"\\nINITIAL STATE VALUES\")\n",
        "print(V)\n",
        "\n",
        "# Synchronous Value Iteration\n",
        "iteration = 0\n",
        "\n",
        "print(\"\\nVALUE ITERATION\")\n",
        "\n",
        "while True:\n",
        "    new_V = V.copy()\n",
        "\n",
        "    for s in non_terminal_states:\n",
        "        q_values = []\n",
        "\n",
        "        for a in actions:\n",
        "            next_state = transition[(s, a)]\n",
        "            q = reward[(s, a)] + gamma * V[next_state]\n",
        "            q_values.append(q)\n",
        "\n",
        "        new_V[s] = max(q_values)\n",
        "\n",
        "    new_V[\"G\"] = 0.0\n",
        "\n",
        "    delta = max(abs(new_V[s] - V[s]) for s in states)\n",
        "    V = new_V\n",
        "    iteration += 1\n",
        "\n",
        "    print(\n",
        "        f\"Iteration {iteration}: \"\n",
        "        f\"V(S0)={V['S0']:.6f}, \"\n",
        "        f\"V(S1)={V['S1']:.6f}, \"\n",
        "        f\"V(G)={V['G']:.6f}, \"\n",
        "        f\"Delta={delta:.6f}\"\n",
        "    )\n",
        "\n",
        "    if delta < theta:\n",
        "        break\n",
        "\n",
        "# Final optimal state values\n",
        "print(\"\\nFINAL OPTIMAL STATE VALUES\")\n",
        "for s in states:\n",
        "    print(f\"V*({s}) = {V[s]:.6f}\")\n",
        "\n",
        "# Calculate action values\n",
        "Q = {}\n",
        "\n",
        "print(\"\\nACTION VALUES Q*(s, a)\")\n",
        "\n",
        "for s in non_terminal_states:\n",
        "    print(f\"\\nState {s}:\")\n",
        "\n",
        "    for a in actions:\n",
        "        next_state = transition[(s, a)]\n",
        "        Q[(s, a)] = (\n",
        "            reward[(s, a)] + gamma * V[next_state]\n",
        "        )\n",
        "        print(f\"Q*({s}, {a}) = {Q[(s, a)]:.6f}\")\n",
        "\n",
        "# Extract optimal policy, including ties\n",
        "print(\"\\nOPTIMAL POLICY\")\n",
        "\n",
        "policy = {}\n",
        "\n",
        "for s in non_terminal_states:\n",
        "    max_q = max(Q[(s, a)] for a in actions)\n",
        "\n",
        "    best_actions = [\n",
        "        a for a in actions\n",
        "        if abs(Q[(s, a)] - max_q) < 1e-9\n",
        "    ]\n",
        "\n",
        "    policy[s] = best_actions\n",
        "    print(f\"Policy({s}) = {best_actions}\")\n",
        "\n",
        "# Bellman Optimality verification\n",
        "print(\"\\nBELLMAN OPTIMALITY VERIFICATION\")\n",
        "\n",
        "for s in non_terminal_states:\n",
        "    max_q = max(Q[(s, a)] for a in actions)\n",
        "    difference = abs(V[s] - max_q)\n",
        "\n",
        "    print(\n",
        "        f\"State: {s}, V*={V[s]:.6f}, \"\n",
        "        f\"max Q*={max_q:.6f}, \"\n",
        "        f\"Difference={difference:.10f}\"\n",
        "    )\n",
        "\n",
        "    assert difference < 1e-8\n",
        "\n",
        "print(\"\\nBellman Optimality verification passed.\")"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "n6obO7JMp1Po",
        "outputId": "3f26a969-bf2f-433b-9a22-91fa3bfe6cb9"
      },
      "execution_count": 1,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "STATES: ['S0', 'S1', 'G']\n",
            "ACTIONS: ['up', 'down', 'left', 'right']\n",
            "Discount factor: 0.9\n",
            "Rewards: Normal = -1, Wall = -1, Goal = +10\n",
            "\n",
            "TRANSITION MAPPING\n",
            "State: S0, Action: up, Next State: S0, Reward: -1\n",
            "State: S0, Action: down, Next State: S0, Reward: -1\n",
            "State: S0, Action: left, Next State: S0, Reward: -1\n",
            "State: S0, Action: right, Next State: S1, Reward: -1\n",
            "State: S1, Action: up, Next State: S1, Reward: -1\n",
            "State: S1, Action: down, Next State: G, Reward: 10\n",
            "State: S1, Action: left, Next State: S0, Reward: -1\n",
            "State: S1, Action: right, Next State: S1, Reward: -1\n",
            "\n",
            "INITIAL STATE VALUES\n",
            "{'S0': 0.0, 'S1': 0.0, 'G': 0.0}\n",
            "\n",
            "VALUE ITERATION\n",
            "Iteration 1: V(S0)=-1.000000, V(S1)=10.000000, V(G)=0.000000, Delta=10.000000\n",
            "Iteration 2: V(S0)=8.000000, V(S1)=10.000000, V(G)=0.000000, Delta=9.000000\n",
            "Iteration 3: V(S0)=8.000000, V(S1)=10.000000, V(G)=0.000000, Delta=0.000000\n",
            "\n",
            "FINAL OPTIMAL STATE VALUES\n",
            "V*(S0) = 8.000000\n",
            "V*(S1) = 10.000000\n",
            "V*(G) = 0.000000\n",
            "\n",
            "ACTION VALUES Q*(s, a)\n",
            "\n",
            "State S0:\n",
            "Q*(S0, up) = 6.200000\n",
            "Q*(S0, down) = 6.200000\n",
            "Q*(S0, left) = 6.200000\n",
            "Q*(S0, right) = 8.000000\n",
            "\n",
            "State S1:\n",
            "Q*(S1, up) = 8.000000\n",
            "Q*(S1, down) = 10.000000\n",
            "Q*(S1, left) = 6.200000\n",
            "Q*(S1, right) = 8.000000\n",
            "\n",
            "OPTIMAL POLICY\n",
            "Policy(S0) = ['right']\n",
            "Policy(S1) = ['down']\n",
            "\n",
            "BELLMAN OPTIMALITY VERIFICATION\n",
            "State: S0, V*=8.000000, max Q*=8.000000, Difference=0.0000000000\n",
            "State: S1, V*=10.000000, max Q*=10.000000, Difference=0.0000000000\n",
            "\n",
            "Bellman Optimality verification passed.\n"
          ]
        }
      ]
    }
  ]
}